# ONLINE LEARNING APPLICATIONS - Course Project

## Riccardo Piantoni

### A.Y. 2025/2026

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np

import matplotlib.pyplot as plt

#from advertising_market_simulator import AdvertisementMarketSimulator
from environment.environment import Environment
from campaigns.highly_non_stationary_campaign import HighlyNonStationaryCampaign
from bidders.primal_dual.primal_dual_bidder import PrimalDualBidder

## Requirement 3: Best-of-both-worlds with multiple campaigns

### Environment

Use the stochastic environment already designed:
- A joint distribution over the highest competing bids for each campaign

Build a highly non-stationary environment. At a high level, it should include:
- A non-stochastic sequence of highest competing bids for each campaign (e.g., sampled from a distribution that changes quickly over time)

Modeling assumptions: A fully adversarial setting in which the competitors' bidders adapt dynamically too, quickly and continuously in time. One could also model the knowledge of the algorithm used from the environment, which plays a bid just slightly greater than the one played by the agent.  

In [2]:
N_TRIALS = 2  # trials for proper curve estimation

BIDS_SPACE = np.linspace(0.0, 1.0, 11)  # possible bids = action space of size 10

# market = AdvertisementMarketSimulator()
environment = Environment(BIDS_SPACE)
environment.generate_random_campaigns(n_campaigns=3, campaign_type=HighlyNonStationaryCampaign, min_competitors=3, max_competitors=5, conflicts_percentage=0.4, seed=47)  # generate a single campaign with random qualities for each advertisers
print(environment)

N_USERS = 100000  # number of ads to be shown sequentially = n_rounds = T

RHO = 0.2  # budget per round -> define either rho or the starting budget B and derive the other (rho = B/T)

-- ENVIRONMENT: --
Bids space: [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]
Number of campaigns: 3
Campaigns:
    -- CAMPAIGN: --
    Type: Highly Non-Stationary
    Description: Competing bids sampled from a highly non-stationary distribution which changes quickly over time.
    Ad qualities: [1. 1. 1. 1. 1. 1.]
    Number of advertisers: 6
    Number of competitors: 5
    -- CAMPAIGN: --
    Type: Highly Non-Stationary
    Description: Competing bids sampled from a highly non-stationary distribution which changes quickly over time.
    Ad qualities: [1. 1. 1. 1.]
    Number of advertisers: 4
    Number of competitors: 3
    -- CAMPAIGN: --
    Type: Highly Non-Stationary
    Description: Competing bids sampled from a highly non-stationary distribution which changes quickly over time.
    Ad qualities: [1. 1. 1. 1.]
    Number of advertisers: 4
    Number of competitors: 3
Conflicts graph (1 = conflicting):
[[0 1 0]
 [1 0 0]
 [0 0 0]]


### Design best-of-both-worlds algorithms with multiple campaigns

- Build a bidding strategy using a primal-dual method with budget constraint.

Hint: Design a primal regret minimizer for the specific problem under study.

!!! For this requirement, you can assume full feedback, i.e., to observe the highest
competing bid for each auction.

In [3]:
def run_trials_combinatorial_clairvoyant(n_trials, n_users, starting_budget, my_valuation, environment: Environment):
    """RHO = rho  # budget per round -> define either rho or the starting budget B and derive the other (rho = B/T)
    STARTING_BUDGET = starting_budget #RHO * N_USERS     # or np.inf for no budget

    MY_VALUATION = my_valuation  # my true valuation of the ads (assumed to be known)"""

    #gamma = []
    expected_clairvoyant_utility = []
    #expected_clairvoyant_payment = []

    auxiliary_bidder = PrimalDualBidder(B=starting_budget, T=n_users, valuations=[my_valuation for _ in range(environment.N_CAMPAIGNS)], environment=environment)  # create a UCB-like bidder with the specified budget, number of rounds, valuation, and valid bids
    # for i in range(environment.N_CAMPAIGNS):
    #     # NOTE: in the multiple campaigns version, this is no longer only the gamma, but also the marginals 
    gamma_i, marginals_i, expected_clairvoyant_utility, expected_clairvoyant_payment_i = environment.compute_clairvoyant_strategy_combinatorial_simple(auxiliary_bidder, campaign_indices=None)
    #     #gamma.append(gamma_i)
    #     expected_clairvoyant_utility.append(expected_clairvoyant_utility_i)
    #     #expected_clairvoyant_payment.append(expected_clairvoyant_payment_i)
    print(gamma_i)
    print(marginals_i)
    print(expected_clairvoyant_utility)
    print(expected_clairvoyant_payment_i)

    #gamma = np.array(gamma)                                                     # shape: (N_CAMPAIGNS)
    expected_clairvoyant_utility = np.array(expected_clairvoyant_utility)       # shape: (N_CAMPAIGNS)
    #expected_clairvoyant_payment = np.array(expected_clairvoyant_payment)       # shape: (N_CAMPAIGNS)

    all_regrets = []
    all_payments = []
    all_pulls = []

    for trial in range(n_trials):
        bidder = PrimalDualBidder(B=starting_budget, T=n_users, valuations=[my_valuation for _ in range(environment.N_CAMPAIGNS)], environment=environment)  # create a UCB-like bidder with the specified budget, number of rounds, valuation, and valid bids

        utilities, my_bids, my_payments, total_wins = environment.simulate_environment(bidder, n_users=n_users, seed=10+trial) # seed=47   

        cumulative_payments = np.cumsum(my_payments, axis=0)
        cumulative_regrets = np.cumsum(expected_clairvoyant_utility - utilities.sum(axis=1), axis=0)    # shape: (1,) - (n_users, 1) = (n_users, 1)  # cumulative regret for each campaign

        all_regrets.append(cumulative_regrets)
        all_payments.append(cumulative_payments)
        all_pulls.append(bidder.get_total_pulls_per_arm())

    all_regrets = np.array(all_regrets)         # shape: (n_trials, n_users, N_CAMPAIGNS)
    all_payments = np.array(all_payments)       # shape: (n_trials, n_users, N_CAMPAIGNS)
    all_pulls = np.array(all_pulls)             # shape: (n_trials, N_CAMPAIGNS, len(BIDS_SPACE))  # number of times each bid was pulled for each campaign
    print(all_regrets.shape, all_payments.shape, all_pulls.shape)

    avg_regrets = all_regrets.mean(axis=0)
    std_regrets = all_regrets.std(axis=0)

    avg_payments = all_payments.mean(axis=0)
    std_payments = all_payments.std(axis=0)

    avg_pulls = all_pulls.mean(axis=0)
    std_pulls = all_pulls.std(axis=0)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    # --------------------------------------------------
    # 1. Cumulative payments
    # --------------------------------------------------
    axes[0].plot(cumulative_payments)
    axes[0].set_xlabel('$t$')
    axes[0].set_ylabel('$\\sum c_t$')
    axes[0].axhline(starting_budget, color='red', label='Budget')
    axes[0].legend()
    axes[0].set_title('Cumulative Payments of UCB-Like Approach (No Budget Constraint)')

    # --------------------------------------------------
    # 2. Cumulative regret
    # --------------------------------------------------
    axes[1].plot(cumulative_regrets)
    axes[1].set_xlabel('$t$')
    axes[1].set_ylabel('$\\sum R_t$')
    axes[1].set_title('Cumulative Regret of UCB-Like Approach (No Budget Constraint)')

    # Hide the unused 6th subplot
    axes[2].axis('off')

    # --------------------------------------------------
    # 3. Chosen bids
    # --------------------------------------------------
    for i in range(auxiliary_bidder.N_CAMPAIGNS):
        axes[3].plot(auxiliary_bidder.bids, avg_pulls[i])
        axes[3].fill_between(
            auxiliary_bidder.bids,
            avg_pulls[i] - std_pulls[i],
            avg_pulls[i] + std_pulls[i],
            alpha=0.3
        )

    axes[3].set_xlabel('$b$')
    axes[3].set_ylabel('$n(b)$')
    axes[3].set_title('Chosen Bids of UCB-Like Approach (No Budget Constraint)')

    # --------------------------------------------------
    # 4. Cumulative payments by campaign
    # --------------------------------------------------
    for i in range(auxiliary_bidder.N_CAMPAIGNS):
        axes[4].plot(np.arange(n_users), avg_payments[:, i])
        axes[4].fill_between(
            np.arange(n_users),
            avg_payments[:, i] - std_payments[:, i],
            avg_payments[:, i] + std_payments[:, i],
            alpha=0.3
        )

    axes[4].set_xlabel('$t$')
    axes[4].set_ylabel('$\\sum c_t$')
    axes[4].axhline(starting_budget, color='red', label='Budget')
    axes[4].legend()
    axes[4].set_title('Cumulative Payments of UCB-Like Approach (No Budget Constraint)')

    # --------------------------------------------------
    # 5. Cumulative regret by campaign
    # --------------------------------------------------
    # for i in range(auxiliary_bidder.N_CAMPAIGNS):
    axes[5].plot(np.arange(n_users), avg_regrets)
    axes[5].fill_between(
        np.arange(n_users),
        avg_regrets - std_regrets,
        avg_regrets + std_regrets,
        alpha=0.3
    )

    axes[5].set_xlabel('$t$')
    axes[5].set_ylabel('$\\sum R_t$')
    axes[5].set_title('Cumulative Regret of UCB-Like Approach (No Budget Constraint)')

    plt.tight_layout()
    plt.show()

In [4]:
STARTING_BUDGET = RHO * N_USERS     # or np.inf for no budget

MY_VALUATION = 0.8  # my true valuation of the ads (assumed to be known)

run_trials_combinatorial_clairvoyant(n_trials=N_TRIALS, n_users=N_USERS, starting_budget=STARTING_BUDGET, my_valuation=MY_VALUATION, environment=environment)

Clairvoyant - combinatorial over campaigns - simple version


ValueError: Competing bids have not been generated yet.